# Naive Bayes Sentiment Analysis

## Install and Import Libraries

In [191]:
import pandas as pd
import nltk as nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score
import matplotlib.pyplot as plt
import re

## Load File, check data

In [152]:
filename = "../datasets/final/mda_shared_preprocessed.csv"
data = pd.read_csv(filename)

In [153]:
data.head()

,doc_id,company_name,filing_type,filing_date,year,quarter,sentence,sentiment
0,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,presently the companys product line includes a...,neutral
1,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,approximately NUM of all sales were generated ...,neutral
2,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,on an ongoing basis we re-evaluate our judgmen...,neutral
3,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,we base our estimates and judgments on our his...,neutral
4,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,actual results may differ from these estimates...,neutral


In [154]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 452390 entries, 0 to 452389
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   doc_id        452390 non-null  str  
 1   company_name  452390 non-null  str  
 2   filing_type   452390 non-null  str  
 3   filing_date   452390 non-null  str  
 4   year          452390 non-null  int64
 5   quarter       452390 non-null  str  
 6   sentence      452390 non-null  str  
 7   sentiment     452390 non-null  str  
dtypes: int64(1), str(7)
memory usage: 27.6 MB


In [155]:
print(f"Loaded : {data.shape[0]:,} documents, {data.shape[1]} columns")
print(f"Columns: {data.columns.tolist()}")
print(f"Companies : {data['company_name'].nunique():,}")
print(f"Year range: {data['year'].min()} – {data['year'].max()}")
print()
print("Filing type counts:")
print(data["filing_type"].value_counts().to_string())

Loaded : 452,390 documents, 8 columns
Columns: ['doc_id', 'company_name', 'filing_type', 'filing_date', 'year', 'quarter', 'sentence', 'sentiment']
Companies : 473
Year range: 2010 – 2025

Filing type counts:
filing_type
10-K    237517
10-Q    214873


## Preprocessing

In [156]:
# Make a copy of the df first in case

df = data.copy()

In [157]:
# Normalise all to lowercase, remove multiple spaces
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

df = df.dropna(subset=["sentence", "sentiment"])
df["clean_sentence"] = df["sentence"].apply(clean_text)
df = df.drop_duplicates(subset=["clean_sentence", "sentiment"])



# # Perform lemmatization - ONLY IF NEEDED
# lemmatizer = nltk.WordNetLemmatizer()

# def lemmatize_text(text):
#     words = re.findall(r"\b\w+\b", str(text).lower())
#     lemmas = [lemmatizer.lemmatize(word) for word in words]
#     return " ".join(lemmas)

# df["sentence_lemma"] = df["clean_sentence"].apply(lemmatize_text)

In [158]:
df.head()

,doc_id,company_name,filing_type,filing_date,year,quarter,sentence,sentiment,clean_sentence
0,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,presently the companys product line includes a...,neutral,presently the companys product line includes a...
1,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,approximately NUM of all sales were generated ...,neutral,approximately num of all sales were generated ...
2,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,on an ongoing basis we re-evaluate our judgmen...,neutral,on an ongoing basis we re-evaluate our judgmen...
3,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,we base our estimates and judgments on our his...,neutral,we base our estimates and judgments on our his...
4,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,actual results may differ from these estimates...,neutral,actual results may differ from these estimates...


In [159]:
print("NaN values:")
print(df.isna().sum())

print("\nEmpty sentence values:")
print((df["sentence"].astype(str).str.strip() == "").sum())

NaN values:
doc_id            0
company_name      0
filing_type       0
filing_date       0
year              0
quarter           0
sentence          0
sentiment         0
clean_sentence    0
dtype: int64

Empty sentence values:
0


In [160]:
df["sentiment"].value_counts()

sentiment
neutral     378386
positive     37564
negative     36440
Name: count, dtype: int64

## Naive Bayes

### Preparing the train/test data split

In [161]:
# Look-ahead bias: Use Filings <= 2022 -> train, 2023 >= -> test.  

df["filing_date"] = pd.to_datetime(df["filing_date"])

# split by filing year cutoff
train_df = df[df["filing_date"].dt.year <= 2022].copy()
test_df  = df[df["filing_date"].dt.year >= 2023].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


Train shape: (341359, 9)
Test shape: (111031, 9)


In [162]:
print("Train date range:", train_df["filing_date"].min(), "to", train_df["filing_date"].max())
print("Test date range:", test_df["filing_date"].min(), "to", test_df["filing_date"].max())

Train date range: 2010-01-08 00:00:00 to 2022-12-23 00:00:00
Test date range: 2023-01-03 00:00:00 to 2025-12-23 00:00:00


In [163]:
print("Train sentiment distribution:")
print(train_df["sentiment"].value_counts(normalize=True))

print("\nTest sentiment distribution:")
print(test_df["sentiment"].value_counts(normalize=True))

# Granted, most of our sentences were identified as neutral

Train sentiment distribution:
sentiment
neutral     0.833243
positive    0.085239
negative    0.081518
Name: proportion, dtype: float64

Test sentiment distribution:
sentiment
neutral     0.846169
negative    0.077573
positive    0.076258
Name: proportion, dtype: float64


In [164]:
X_train = train_df["clean_sentence"].astype(str)
y_train = train_df["sentiment"]

X_test = test_df["clean_sentence"].astype(str)
y_test = test_df["sentiment"]


# # If using lemmatized version
# X_train = train_df["sentence_lemma"].astype(str)
# y_train = train_df["sentiment"]

# X_test = test_df["sentence_lemma"].astype(str)
# y_test = test_df["sentiment"]


### 1. Bag of Words (Baseline)

In [165]:
vectorizer = CountVectorizer() # Init BoW Matrix


X_train_vector = vectorizer.fit_transform(X_train) # Fit and transform the training data
X_test_vector = vectorizer.transform(X_test) # Transform the testing data

In [166]:
nb_bow = MultinomialNB() # Init

nb_bow.fit(X_train_vector, y_train)


,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [167]:
y_pred = nb_bow.predict(X_test_vector)

# Print the classification report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    negative       0.39      0.68      0.49      8613
     neutral       0.95      0.88      0.92     93951
    positive       0.46      0.48      0.47      8467

    accuracy                           0.84    111031
   macro avg       0.60      0.68      0.63    111031
weighted avg       0.87      0.84      0.85    111031



### Feature Engineering

For this project, we plan on using implementing the follwoing features:

1. **Unigram + Bigrams through `CountVectorizer`'s `ngram_range`**

- This is because sentiments in financial text often rely on short phrases instead of just single words e.g. `strong demand`

- Another thing would be for negation, which Bigrams are able to capture e.g. `not profitable` is different from `profitable` in a unigram

2. **Domain keyword features**

- We would attempt to use the `Loughran-McDonald` dictionary to add count features of positive and negative words in a sentence to improve on the Naive Bayes model. We explicitly decided on the `Loughran-McDonald` dictionary as it is more suitable than a general lexicon.

3. **TF-IDF**

- Aside from using raw count-based features, we will also experiment with `TF-IDF` by placing more weight on rarer words, which might help improve sentence classification performance.

4. **ComplementNB**

- Due to an imbalanced dataset, we believe that using `ComplementNB` instead of the standard `MultinomialNB` would help in improving performance of classification



#### Adding the Loughran-McDonald counts

In [168]:
# Creating the dictionary 

lm_dict = pd.read_csv("../datasets/final/Loughran-McDonald_MasterDictionary_1993-2024.csv")
print(lm_dict[["Word", "Positive", "Negative"]].tail())


# We make all words to lower to try and match the sentences

positive_words = set(
    lm_dict.loc[lm_dict["Positive"] > 0, "Word"]
    .str.lower()
    .str.strip()
)

negative_words = set(
    lm_dict.loc[lm_dict["Negative"] > 0, "Word"]
    .str.lower()
    .str.strip()
)

print("Positive words:", len(positive_words))
print("Negative words:", len(negative_words))

            Word  Positive  Negative
86548     ZYGOTE         0         0
86549    ZYGOTES         0         0
86550    ZYGOTIC         0         0
86551  ZYMURGIES         0         0
86552    ZYMURGY         0         0
Positive words: 347
Negative words: 2345


In [169]:
# Function to count the words and create columns

def lm_count_features(text):
    words = re.findall(r"\b\w+\b", str(text).lower())
        
    pos_count = sum(word in positive_words for word in words)
    neg_count = sum(word in negative_words for word in words)
    
    return pd.Series({
        "lm_pos_count": pos_count,
        "lm_neg_count": neg_count
    })

In [170]:
train_features = train_df[["clean_sentence", "sentiment"]].copy()
test_features = test_df[["clean_sentence", "sentiment"]].copy()

train_lm = train_features["clean_sentence"].apply(lm_count_features)
test_lm = test_features["clean_sentence"].apply(lm_count_features)

train_features = pd.concat([train_features, train_lm], axis=1)
test_features = pd.concat([test_features, test_lm], axis=1)

In [171]:
print(train_features.head())
print(test_features.head())

                                      clean_sentence sentiment  lm_pos_count  \
0  presently the companys product line includes a...   neutral             1   
1  approximately num of all sales were generated ...   neutral             0   
2  on an ongoing basis we re-evaluate our judgmen...   neutral             0   
3  we base our estimates and judgments on our his...   neutral             0   
4  actual results may differ from these estimates...   neutral             0   

   lm_neg_count  
0             0  
1             0  
2             1  
3             0  
4             0  
                                        clean_sentence sentiment  \
106  management does not believe that the resolutio...   neutral   
107  we cannot assure you that we will not be the s...   neutral   
108  we generally place orders for products with ou...   neutral   
109  these estimates may be significantly different...   neutral   
110  in the event that subsequent orders fall short...   neutral   

  

In [172]:
# Create the new train and test datasets

X_train_text = train_features["clean_sentence"].astype(str)
y_train_text = train_features["sentiment"]

X_test_text = test_features["clean_sentence"].astype(str)
y_test_text = test_features["sentiment"]

### 2. "Fine-tuned" BoW (Unigrams + Bigrams, Domain Keyword features)

In [173]:
vectorizer_finetune = CountVectorizer(ngram_range=(1,2))

X_train_bow_ft = vectorizer_finetune.fit_transform(X_train_text)
X_test_bow_ft = vectorizer_finetune.transform(X_test_text)

In [174]:
# Take the numerical counts

X_train_lm = train_features[["lm_pos_count", "lm_neg_count"]]
X_test_lm = test_features[["lm_pos_count", "lm_neg_count"]]

In [175]:
# Combine the features
from scipy.sparse import hstack, csr_matrix

X_train_final = hstack([X_train_bow_ft, csr_matrix(X_train_lm.values)])
X_test_final = hstack([X_test_bow_ft, csr_matrix(X_test_lm.values)])


In [176]:
# Train the model

nb_bow_fine = MultinomialNB() # Init

nb_bow_fine.fit(X_train_final, y_train_text)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [177]:
y_pred_ft = nb_bow_fine.predict(X_test_final)

print("Accuracy:", accuracy_score(y_test_text, y_pred_ft))
print(classification_report(y_test_text, y_pred_ft))

Accuracy: 0.8652178220496979
              precision    recall  f1-score   support

    negative       0.44      0.69      0.54      8613
     neutral       0.95      0.91      0.93     93951
    positive       0.58      0.51      0.54      8467

    accuracy                           0.87    111031
   macro avg       0.66      0.70      0.67    111031
weighted avg       0.88      0.87      0.87    111031



### 3. TF-IDF Vectorizer (Unigrams + Bigrams)

In [178]:
tfidfvectorizer = TfidfVectorizer(ngram_range=(1, 2))

X_train_tfidf = tfidfvectorizer.fit_transform(X_train_text)
X_test_tfidf = tfidfvectorizer.transform(X_test_text)

In [179]:
nb_tfidf = MultinomialNB()

nb_tfidf.fit(X_train_tfidf, y_train_text)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [180]:
y_pred_tfidf = nb_tfidf.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_text, y_pred_tfidf))
print(classification_report(y_test_text, y_pred_tfidf))
print(confusion_matrix(y_test_text, y_pred_tfidf))

Accuracy: 0.8625338869324783
              precision    recall  f1-score   support

    negative       0.83      0.13      0.23      8613
     neutral       0.86      1.00      0.93     93951
    positive       0.90      0.09      0.17      8467

    accuracy                           0.86    111031
   macro avg       0.86      0.41      0.44    111031
weighted avg       0.86      0.86      0.81    111031

[[ 1151  7401    61]
 [   84 93839    28]
 [  144  7545   778]]


### 4. TF-IDF Vectorizer (Unigrams + Bigrams, Domain Keyword features)

In [181]:
tfidfvectorizer_ft = TfidfVectorizer(ngram_range=(1, 2))

X_train_tfidf_ft = tfidfvectorizer_ft.fit_transform(X_train_text)
X_test_tfidf_ft = tfidfvectorizer_ft.transform(X_test_text)

In [182]:
X_train_tfidf_final_ft = hstack([X_train_tfidf_ft, csr_matrix(X_train_lm.values)])
X_test_tfidf_final_ft = hstack([X_test_tfidf_ft, csr_matrix(X_test_lm.values)])

In [183]:
nb_tfidf_ft = MultinomialNB()

nb_tfidf_ft.fit(X_train_tfidf_final_ft, y_train_text)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [184]:
y_pred_tfidf_ft = nb_tfidf_ft.predict(X_test_tfidf_final_ft)

print("Accuracy:", accuracy_score(y_test_text, y_pred_tfidf_ft))
print(classification_report(y_test_text, y_pred_tfidf_ft))
print(confusion_matrix(y_test_text, y_pred_tfidf_ft))

Accuracy: 0.8623357440714755
              precision    recall  f1-score   support

    negative       0.86      0.12      0.21      8613
     neutral       0.86      1.00      0.93     93951
    positive       0.88      0.10      0.18      8467

    accuracy                           0.86    111031
   macro avg       0.87      0.41      0.44    111031
weighted avg       0.86      0.86      0.81    111031

[[ 1030  7512    71]
 [   51 93859    41]
 [  112  7498   857]]


### 4. ComplementNB

In [185]:
cnb_bow = ComplementNB()
cnb_bow.fit(X_train_final, y_train_text)


,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueOnly used in edge case with a single class in the training set.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. Not used.",None
,"norm norm: bool, default=FalseWhether or not a second normalization of the weights is performed. Thedefault behavior mirrors the implementations found in Mahout and Weka,which do not follow the full algorithm described in Table 9 of thepaper.",False


In [186]:
y_pred_cnb_bow = cnb_bow.predict(X_test_final)

print("Accuracy:", accuracy_score(y_test_text, y_pred_cnb_bow))
print(classification_report(y_test_text, y_pred_cnb_bow))
print(confusion_matrix(y_test_text, y_pred_cnb_bow))

Accuracy: 0.8649656402266034
              precision    recall  f1-score   support

    negative       0.44      0.70      0.54      8613
     neutral       0.95      0.91      0.93     93951
    positive       0.59      0.54      0.56      8467

    accuracy                           0.86    111031
   macro avg       0.66      0.72      0.68    111031
weighted avg       0.89      0.86      0.87    111031

[[ 6000  1960   653]
 [ 5899 85464  2588]
 [ 1607  2286  4574]]


### 5. ComplementNB with TF-IDF + Domain Keyword features

In [187]:
cnb_tfidf_ft = ComplementNB()
cnb_tfidf_ft.fit(X_train_tfidf_final_ft, y_train_text)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueOnly used in edge case with a single class in the training set.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. Not used.",None
,"norm norm: bool, default=FalseWhether or not a second normalization of the weights is performed. Thedefault behavior mirrors the implementations found in Mahout and Weka,which do not follow the full algorithm described in Table 9 of thepaper.",False


In [188]:
y_pred_cnb_tfidf_ft = cnb_tfidf_ft.predict(X_test_tfidf_final_ft)

print("Accuracy:", accuracy_score(y_test_text, y_pred_cnb_tfidf_ft))
print(classification_report(y_test_text, y_pred_cnb_tfidf_ft))
print(confusion_matrix(y_test_text, y_pred_cnb_tfidf_ft))

Accuracy: 0.8882114004197026
              precision    recall  f1-score   support

    negative       0.63      0.43      0.51      8613
     neutral       0.91      0.97      0.94     93951
    positive       0.69      0.41      0.51      8467

    accuracy                           0.89    111031
   macro avg       0.74      0.60      0.65    111031
weighted avg       0.87      0.89      0.88    111031

[[ 3695  4289   629]
 [ 1519 91486   946]
 [  697  4332  3438]]


## Pick a model

In [190]:
y_true = y_test_text   # or y_test if that's the one you used consistently

model_preds = {
    "Baseline BoW + MNB": y_pred,
    "BoW + LM + MNB": y_pred_ft,
    "TF-IDF + MNB": y_pred_tfidf,
    "TF-IDF + LM + MNB": y_pred_tfidf_ft,
    "BoW + LM + CNB": y_pred_cnb_bow,
    "TF-IDF + LM + CNB": y_pred_cnb_tfidf_ft,
}

In [196]:
results = []

for model_name, y_pred_model in model_preds.items():
    results.append({
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred_model),
        "macro_precision": precision_score(y_true, y_pred_model, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred_model, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred_model, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred_model, average="weighted"),
    })

results_df = pd.DataFrame(results)
# results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False)

results_df

,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1
0,Baseline BoW + MNB,0.837190,0.601458,0.680166,0.627189,0.850000
1,BoW + LM + MNB,0.865218,0.657164,0.704477,0.670811,0.871954
2,TF-IDF + MNB,0.862534,0.864873,0.408110,0.440938,0.813905
3,TF-IDF + LM + MNB,0.862336,0.869971,0.406608,0.439087,0.813311
4,BoW + LM + CNB,0.864966,0.660734,0.715501,0.678346,0.872435
5,TF-IDF + LM + CNB,0.888211,0.741602,0.602938,0.653925,0.876198


Based on F1 score, Model 5 (TF-IDF + LM + CNB) performed the best

In [199]:
# Obtain per-class F1 Score for each model

import pandas as pd
from sklearn.metrics import classification_report

# use the same y_true for all models
y_true = y_test_text   # or y_test if that's the one you used consistently

model_preds = {
    "Baseline BoW + MNB": y_pred,
    "BoW + LM + MNB": y_pred_ft,
    "TF-IDF + MNB": y_pred_tfidf,
    "TF-IDF + LM + MNB": y_pred_tfidf_ft,
    "BoW + LM + CNB": y_pred_cnb_bow,
    "TF-IDF + LM + CNB": y_pred_cnb_tfidf_ft,
}

# collect per-class scores
per_class_results = []

for model_name, y_pred_model in model_preds.items():
    report = classification_report(y_true, y_pred_model, output_dict=True, zero_division=0)
    
    for label in ["negative", "neutral", "positive"]:
        per_class_results.append({
            "model": model_name,
            "class": label,
            "precision": report[label]["precision"],
            "recall": report[label]["recall"],
            "f1": report[label]["f1-score"],
        })

per_class_df = pd.DataFrame(per_class_results)
per_class_df

,model,class,precision,recall,f1
0,Baseline BoW + MNB,negative,0.387557,0.677697,0.493115
1,Baseline BoW + MNB,neutral,0.952023,0.884120,0.916816
2,Baseline BoW + MNB,positive,0.464794,0.478682,0.471636
3,BoW + LM + MNB,negative,0.443834,0.693603,0.541295
4,BoW + LM + MNB,neutral,0.952108,0.913274,0.932287
5,BoW + LM + MNB,positive,0.575550,0.506555,0.538853
6,TF-IDF + MNB,negative,0.834663,0.133635,0.230384
7,TF-IDF + MNB,neutral,0.862610,0.998808,0.925726
8,TF-IDF + MNB,positive,0.897347,0.091886,0.166702
9,TF-IDF + LM + MNB,negative,0.863370,0.119587,0.210075


In [200]:
best_per_class_f1 = (
    per_class_df.loc[per_class_df.groupby("class")["f1"].idxmax()]
    .sort_values("class")
    .reset_index(drop=True)
)

best_per_class_f1

,model,class,precision,recall,f1
0,BoW + LM + CNB,negative,0.444247,0.696621,0.542520
1,TF-IDF + LM + CNB,neutral,0.913882,0.973763,0.942873
2,BoW + LM + CNB,positive,0.585285,0.540215,0.561847


In [193]:
# Save the model for use in case we need it

import pickle

with open("nb_model.pkl", "wb") as f:
    pickle.dump(cnb_tfidf_ft, f)

with open("nb_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidfvectorizer_ft, f)

with open("positive_words.pkl", "wb") as f:
    pickle.dump(positive_words, f)

with open("negative_words.pkl", "wb") as f:
    pickle.dump(negative_words, f)